# Experiment

## Import libraries

In [15]:
import pandas as pd

file_path = "brd_vietstock_2000_2025.csv"

df = pd.read_csv(file_path)


# Set indicator names as lowercase with underscores
# df["Chỉ tiêu"] = (
#     df["Chỉ tiêu"].str.lower().str.replace(",", "").str.replace(" ", "_")
# )

column_name = "Chỉ tiêu"

df[column_name] = (
    df[column_name]
    .str.lower()
    .str.replace(
        r"[^a-z0-9_\s-]", "", regex=True
    )  # remove everything except letters, numbers, underscore, space
    .str.replace(r"[\s-]+", "_", regex=True)  # replace any whitespace with underscore
)

id_vars = ["Chỉ tiêu", "Đơn vị tính"]

# Melt from wide to long format
df = df.melt(
    id_vars=id_vars,
    var_name="month_str",
    value_name="value",
)

# Clean numeric values
df["value"] = df["value"].astype(str).str.replace(",", "", regex=False)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

# Extract year and month
df["date"] = pd.to_datetime(df["month_str"], errors="coerce")

# Drop rows where date couldn't be parsed
df = df.dropna(subset=["date"])

# Extract numeric year, month
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year

# Use pivot_table with first() to handle duplicates
df = df.pivot_table(
    index=["year", "month"],
    columns=id_vars[0],
    values="value",
    aggfunc="first",
).reset_index()

# Sort by year and month
df = df.sort_values(["year", "month"]).reset_index(drop=True)

# Fill missing values with 0
df.fillna(0, inplace=True)

df

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_21980\1118037570.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["month_str"], errors="coerce")


Chỉ tiêu,year,month,enterprises_completing_dissolution,enterprises_resuming_operations,enterprises_temporarily_suspended_awaiting_dissolution,newly_established_enterprises,registered_capital,registered_labor
0,2014,1,1028.0,2375.0,0.0,6866.0,43.72,0.00
1,2014,2,863.0,0.0,0.0,4003.0,19.18,0.00
2,2014,3,690.0,982.0,0.0,7487.0,35.08,0.00
3,2014,4,694.0,0.0,0.0,7373.0,45.43,0.00
4,2014,5,627.0,1131.0,0.0,5499.0,30.22,0.00
...,...,...,...,...,...,...,...,...
131,2024,12,2345.0,8843.0,19886.0,9996.0,96.41,95.75
132,2025,1,3493.0,22794.0,3500.0,10653.0,94.07,81.54
133,2025,2,1737.0,7053.0,2971.0,10128.0,136.38,59143.00
134,2025,3,9122.0,2137.0,4899.0,15619.0,126.30,87517.00


In [16]:
df.columns

Index(['year', 'month', 'enterprises_completing_dissolution',
       'enterprises_resuming_operations',
       'enterprises_temporarily_suspended_awaiting_dissolution',
       'newly_established_enterprises', 'registered_capital',
       'registered_labor'],
      dtype='object', name='Chỉ tiêu')